In [4]:
!pip install transformers datasets

     ---------------------------------------- 10.4/10.4 MB 1.4 MB/s eta 0:00:00
     -------------------------------------- 527.0/527.0 kB 1.6 MB/s eta 0:00:00
     -------------------------------------- 646.8/646.8 kB 1.8 MB/s eta 0:00:00
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl (158 kB)
     ---------------------------------------- 2.7/2.7 MB 1.6 MB/s eta 0:00:00
     -------------------------------------- 56.0/56.0 kB 162.8 kB/s eta 0:00:00
     -------------------------------------- 341.4/341.4 kB 1.2 MB/s eta 0:00:00
     ---------------------------------------- 27.3/27.3 MB 1.3 MB/s eta 0:00:00
     -------------------------------------- 120.0/120.0 kB 1.4 MB/s eta 0:00:00
     -------------------------------------- 144.5/144.5 kB 1.4 MB/s eta 0:00:00
     -------------------------------------- 462.9/462.9 kB 1.6 MB/s eta 0:00:00
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl (3.7 MB)
  Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
  Using cached anyio-4.13

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googletrans 4.0.0rc1 requires httpx==0.13.3, but you have httpx 0.25.1 which is incompatible.

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import torch
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm

ModuleNotFoundError: No module named 'transformers'

In [15]:
import pandas as pd
import torch
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("GROUBD_TRUTH_WITH_FINAL_LABEL.csv", index_col=0)
print(f"✅ Loaded {len(df)} reviews")

# Save original labels
df["original_final_label"] = df["final_label"]

# Drop the 4 label columns
columns_to_drop = ['label_1', 'label_2', 'label_3', 'final_label']
df.drop(columns=columns_to_drop, inplace=True)
print(f"Dataset shape: {df.shape}")

# Load model with the required transformers pipeline API
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
device = 0 if torch.cuda.is_available() else -1
sentiment_pipeline = pipeline(
    "text-classification",
    model=model_name,
    device=device,
    truncation=True,
    max_length=512
)
print(" Model loaded successfully with transformers.pipeline()!")

print("STEP 3: Making predictions on 200 reviews")
print("=" * 60)

label_map = {
    "LABEL_0": "negative",
    "LABEL_1": "neutral",
    "LABEL_2": "positive"
}

def predict_sentiment(text, neutral_threshold=0.55, confidence_margin=0.15):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return "neutral"
    
    outputs = sentiment_pipeline(text, top_k=None)[0]
    mapped_scores = {
        label_map.get(item["label"], item["label"].lower()): item["score"]
        for item in outputs
    }
    ranked_scores = sorted(mapped_scores.items(), key=lambda item: item[1], reverse=True)
    best_label, best_score = ranked_scores[0]
    second_score = ranked_scores[1][1] if len(ranked_scores) > 1 else 0.0
    
    # Confidence-based transformation: route uncertain non-neutral predictions to neutral.
    if best_label != "neutral" and best_score < neutral_threshold and (best_score - second_score) < confidence_margin:
        return "neutral"
    
    return best_label

predictions = []
total = len(df)

for i, row in df.iterrows():
    # Show progress every 20 reviews
    if (i + 1) % 20 == 0 or i == 0:
        print(f"⏳ Processing: {i+1}/{total} reviews...")
    
    pred = predict_sentiment(row["review_text"])
    predictions.append(pred)

df["predicted_label"] = predictions
print(f"✅ All {total} reviews processed!")

print("\n" + "=" * 60)
print("STEP 4: Sample results (first 10 reviews)")
print("=" * 60)

for i in range(min(10, len(df))):
    review_text = df.iloc[i]['review_text']
    if len(str(review_text)) > 60:
        review_text = str(review_text)[:60] + "..."
    print(f"\n{i+1}. Review: {review_text}")
    print(f"   Predicted by RoBERTa: {df.iloc[i]['predicted_label']}")
    print(f"   Original label:      {df.iloc[i]['original_final_label']}")

print("\n" + "=" * 60)
print("STEP 5: Model Evaluation (Accuracy)")
print("=" * 60)

# Calculate accuracy
accuracy = accuracy_score(df["original_final_label"], df["predicted_label"])
print(f"\n🎯 Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\n Detailed Classification Report:")
print(classification_report(df["original_final_label"], df["predicted_label"]))

print("\n Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix(df["original_final_label"], df["predicted_label"]))

print("\n" + "=" * 60)
print("STEP 6: Saving results")
print("=" * 60)

# Save to CSV
df.to_csv("roberta_predictions.csv", index=False)
print("✅ Results saved to: roberta_predictions.csv")

print("\n" + "=" * 60)
print("STEP 7: Prediction Distribution")
print("=" * 60)
print(df["predicted_label"].value_counts())

print("\n" + "=" * 60)
print("✅ ALL DONE!")
print("=" * 60)

✅ Loaded 200 reviews
Dataset shape: (200, 17)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Model loaded successfully!
STEP 3: Making predictions on 200 reviews
⏳ Processing: 1/200 reviews...
⏳ Processing: 20/200 reviews...
⏳ Processing: 40/200 reviews...
⏳ Processing: 60/200 reviews...
⏳ Processing: 80/200 reviews...
⏳ Processing: 100/200 reviews...
⏳ Processing: 120/200 reviews...
⏳ Processing: 140/200 reviews...
⏳ Processing: 160/200 reviews...
⏳ Processing: 180/200 reviews...
⏳ Processing: 200/200 reviews...
✅ All 200 reviews processed!

STEP 4: Sample results (first 10 reviews)

1. Review: `
   Predicted by RoBERTa: neutral
   Original label:      neutral

2. Review: incredibly fun and great replayability (i think that's how y...
   Predicted by RoBERTa: positive
   Original label:      positive

3. Review: I Monster my Hunter till I Wilds
   Predicted by RoBERTa: neutral
   Original label:      neutral

4. Review: This is a really good game if you love Zombie Apocolypse gam...
   Predicted by RoBERTa: positive
   Original label:      positive

5. Review: love the game 